# 03 - Multi-Tool
> Claude escolhendo entre multiplas ferramentas

**Modulo:** `03_tool_use` | **Feito com:** [Claude](https://claude.ai) (Anthropic)

---


In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
client = anthropic.Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
HAIKU  = 'claude-haiku-4-5-20251001'
SONNET = 'claude-sonnet-4-5'

def ask(prompt, system=None, model=HAIKU, max_tokens=1024):
    kw = dict(model=model, max_tokens=max_tokens,
              messages=[{'role':'user','content':prompt}])
    if system: kw['system'] = system
    return client.messages.create(**kw).content[0].text

print('Pronto!')

In [ ]:
import math, datetime, random, string

TOOLS=[
    {'name':'calcular','description':'Calcula expressoes.',
     'input_schema':{'type':'object','properties':{'expr':{'type':'string'}},'required':['expr']}},
    {'name':'data_hora','description':'Data e hora atual.',
     'input_schema':{'type':'object','properties':{}}},
    {'name':'gerar_senha','description':'Gera senha segura.',
     'input_schema':{'type':'object','properties':{'n':{'type':'integer'}}}}
]

FUNS={
    'calcular': lambda expr: str(eval(expr,{'__builtins__':{},'math':math},{})),
    'data_hora': lambda: datetime.datetime.now().strftime('%d/%m/%Y %H:%M'),
    'gerar_senha': lambda n=16: ''.join(random.choice(string.ascii_letters+string.digits+'!@#') for _ in range(n))
}

def agente(q):
    msgs=[{'role':'user','content':q}]
    while True:
        r=client.messages.create(model=HAIKU,max_tokens=512,tools=TOOLS,messages=msgs)
        if r.stop_reason=='end_turn': return r.content[0].text
        msgs.append({'role':'assistant','content':r.content})
        res=[]
        for b in r.content:
            if b.type=='tool_use':
                print(f'  tool: {b.name}({b.input})')
                res.append({'type':'tool_result','tool_use_id':b.id,'content':str(FUNS[b.name](**b.input))})
        msgs.append({'role':'user','content':res})

print(agente('Que horas sao agora, gere uma senha de 20 chars e calcule 2**10.'))

## Exercicios
> Complete os exercicios abaixo.


In [ ]:
# Seu codigo aqui


## Proximos passos
- Proximo notebook do modulo
- [docs.anthropic.com](https://docs.anthropic.com)
